<a href="https://colab.research.google.com/github/LourdAbuHadid1/IE-332-Group-Assignments/blob/main/A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipython-sql
%load_ext sql
%sql sqlite://

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.8 MB/s eta 0:00:00


In [ ]:
%%sql
# Run this setup cell as provided. No package installation is needed.
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

DB_PATH = Path("boilermaker_brews.db")

if not DB_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place boilermaker_brews.db in the notebook's working folder."
        ) from exc
    print("Choose boilermaker_brews.db from the course files.")
    files.upload()

if not DB_PATH.exists():
    raise FileNotFoundError("boilermaker_brews.db was not uploaded.")

# These exercises only read data; protect the uploaded database from changes.
conn = sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)


class SQLResult(list):
    """Query rows and the information needed by the provided self-checks."""
    def __init__(self, frame, sql):
        super().__init__(frame.itertuples(index=False, name=None))
        self.keys = list(frame.columns)
        self.frame = frame
        self.sql = sql


def _sql_magic(line, cell):
    """Run SQL and display its result; %%sql q1 << also saves it as q1."""
    parts = line.split()
    if parts and (
        not parts[0].isidentifier()
        or len(parts) > 2
        or (len(parts) == 2 and parts[1] != "<<")
    ):
        raise ValueError("Use %%sql or a capture header such as %%sql q1 <<.")
    target = parts[0] if parts else None
    if target:
        # A failed rerun must not leave an earlier answer available to the check.
        get_ipython().user_ns[target] = None
    with conn:
        cursor = conn.execute(cell)
        if cursor.description is None:
            raise ValueError("Write a SELECT or WITH query below the %%sql header.")
        frame = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
    if target:
        get_ipython().user_ns[target] = SQLResult(frame, cell)
    display(frame)


get_ipython().register_magic_function(_sql_magic, "cell", "sql")


def _sql_line_magic(line):
    _sql_magic("", line)


get_ipython().register_magic_function(_sql_line_magic, "line", "sql")
print(f"Connected to {DB_PATH.name}. The %%sql command is ready.")


Connected to boilermaker_brews.db. The %%sql command is ready.


In [4]:
%%sql
#Q1
#Revenue and units by category. Show the same three-column result (category, total revenue, and units sold) twice:
#first ordered by revenue from highest to lowest, then ordered by units from highest to lowest.
#You may change only the ORDER BY. Which two categories reverse their relative order, and how do their totals explain the reversal?

#Sorting by revenue
SELECT category, sum(o.unit_price*o.quantity) as revenue, sum(o.quantity) as units
FROM products as p
JOIN order_items as o on p.product_id = o.product_id
ORDER BY revenue DESC

#Sorting by units
SELECT category, sum(o.unit_price*o.quantity) as revenue, sum(o.quantity) as units
FROM products as p
JOIN order_items as o on p.product_id = o.product_id
ORDER BY units DESC

 * sqlite://
(sqlite3.OperationalError) near "#Q1": syntax error
[SQL: #Q1 
#Revenue and units by category. Show the same three-column result (category, total revenue, and units sold) twice:
#first ordered by revenue from highest to lowest, then ordered by units from highest to lowest.
#You may change only the ORDER BY. Which two categories reverse their relative order, and how do their totals explain the reversal?

#Sorting by revenue
SELECT category, sum(o.unit_price*o.quantity) as revenue, sum(o.quantity) as units
FROM products as p
JOIN order_items as o on p.product_id = o.product_id
ORDER BY revenue DESC

#Sorting by units
SELECT category, sum(o.unit_price*o.quantity) as revenue, sum(o.quantity) as units
FROM products as p
JOIN order_items as o on p.product_id = o.product_id
ORDER BY units DESC]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
